## Model verification figures for Joint Western and Eastern Snow Conference 2026

Plots of snow depth verification using SNOTEL and ASO  
Basins and water year coverage: Animas, Yampa, Jordan from WY 2022-2024  
Uses studio conda env


In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

import seaborn as sns

In [ ]:
%reload_ext autoreload
%autoreload 2

### Read in stats files into a dict


In [ ]:
# Read in the stats generated from diagnostic_statistics_snotel_sites.ipynb
basins = ['animas', 'yampa', 'jordan']
thisvar = 'thickness' # isnobal variable

stats_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/thp/statistics'
stats_dict = dict()
for basin in basins:
    stats_fn = h.fn_list(stats_dir, f'{basin}/{basin}*stats*.csv')[0]
    print(stats_fn)
    df = pd.read_csv(stats_fn, index_col=["site", "water_year"])
    stats_dict[basin] = df

### Plot SNOTEL KGE barplots by basin, year + elevation plot of SNOTEL sites


In [ ]:
# Try to combine the dict into a single big df
big_df = pd.DataFrame()
for basin, df in stats_dict.items():
    df_copy = copy.deepcopy(df)
    df_copy['basin'] = basin
    big_df = pd.concat([big_df, df_copy])

big_df

In [ ]:
# Remove multi-indexing for plotting
big_df.reset_index(inplace=True)
big_df

In [ ]:
# Loop through each water year and plot KGE values as barplots, grouped by basin, colored by site
fig, axa = plt.subplots(1, 3, figsize=(20,4), sharey=True)
for ax, wy in zip(axa.flatten(), big_df['water_year'].unique()):
    # Only plot the SNOTEL sites in the basin, don't make space for the sites in other basins
    sns.barplot(data=big_df[big_df['water_year'] == wy], x='basin', y='kge', hue='site', ec='k', dodge='auto', ax=ax)
    ax.set_title(f'WY {wy}')
    ax.set_ylabel('KGE')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Animas', 'Yampa', 'Jordan'])
    # Add horizontal line at KGE=0.41
    ax.axhline(0.41, color='red', linestyle='--', label='KGE=0.41')
    # Add zeroline for reference
    ax.axhline(0, color='black', linestyle='-')
    # Turn off the legend except if you're on the last plot
    if ax != axa.flatten()[-1]:
        ax.legend_.remove()
    else:
        # format the legend so that it is outside the axes and is centered below the plots
        ax.legend(loc='upper center', bbox_to_anchor=(-0.75, -0.15), ncol=6)
    ax.grid(axis='y', color='gray', linestyle='--', alpha=0.5)
# Adjust subplots to snug them together horizontally
plt.subplots_adjust(wspace=0.05)



In [ ]:
fig, axa = plt.subplots(1, 3, figsize=(16, 3.5), sharey=True)
palette_names = ['mako', 'husl', 'Paired']
plotting_df = big_df.dropna(subset=['kge']).copy()
max_sites = max(df['site'].nunique() for _, df in plotting_df.groupby('basin'))

# Reorder basin plotting order (defaults to alphabetical)
basin_order = ['animas', 'yampa', 'jordan']
plotting_df['basin'] = pd.Categorical(plotting_df['basin'], categories=basin_order, ordered=True)
# Sort values first by basin, then by site (for alphabetical site ordering)
plotting_df = plotting_df.sort_values(['basin', 'site'])

for ax, palette_name, (basin, df_basin) in zip(axa, palette_names, plotting_df.groupby('basin', sort=False, observed=False)):
    n_sites = df_basin['site'].nunique()
    palette = sns.color_palette(palette_name, n_colors=n_sites)
    hue_order = sorted(df_basin['site'].unique()) # ensure sites are ordered alphabetically even if missing data in some years
    sns.barplot(data=df_basin, x='water_year', y='kge', hue='site', hue_order=hue_order,
                ec='k', ax=ax, width=0.8 * n_sites / max_sites, palette=palette)
    ax.set_title(basin.capitalize())
    # ax.axhline(-0.41, color='red', linestyle='--', label='KGE=-0.41')
    ax.axhline(0, color='black', linestyle='-')
    ax.grid(axis='y', color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
    ax.set_xlabel('')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['WY 2022', 'WY 2023', 'WY 2024'])
    ax.set_ylabel('KGE')

plt.subplots_adjust(wspace=0.1)

# Save to file
outdir = '/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update/plot_eb_terms/figures'
outname = f'wsc_{thisvar}_snotel_kge_barplots.png'
print(f'Saving figure to {outdir}/{outname}')
plt.savefig(f'{outdir}/{outname}', dpi=300, bbox_inches='tight')

In [ ]:
# Split KGE values into its component parts


### WSC - ASO radial plot Yampa 2024


In [ ]:
sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc
import plot_sdd_radial as psr

In [ ]:
import xarray as xr
from pathlib import PurePath, Path
# Generate radial plots of snow depth differences
def crank_snow_depth_diff_radial(basin, WY,
                                 workdir='/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/ASO_diffs',
                                 script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts',
                                 outdir='/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update/plot_eb_terms/figures/depth_diff',
                                 original=True, verbose=True, num_aspect_bins=16, palette='PuOr'):
    # Locate snow depth difference files
    if original:
        depth_diff_fns = h.fn_list(workdir, f'{basin}*wy{WY}*diff*depth_original.nc')
        # Only run for the iSnobal diffs removing fns if 'ua' or 'nwm' in filename
        depth_diff_fns = [h for h in depth_diff_fns if 'ua' not in h and 'nwm' not in h]
    else:
        # This is for the uniform reprojection diffs to match the terrain data grid
        depth_diff_fns = h.fn_list(workdir, f'{basin}*wy{WY}*diff*depth_uniformreproj.nc')
        # Only run for UA and NWM diffs removing fns if 'ua' or 'nwm' in filename
        depth_diff_fns = [h for h in depth_diff_fns if 'ua' in h or 'nwm' in h]
    if not depth_diff_fns:
        print(f'No depth diff files found for {basin} WY {WY}')
        return
    else:
        for depth_diff_fn in depth_diff_fns:
            if verbose:
                print(f'{depth_diff_fn}\n')
            dt = PurePath(depth_diff_fn).stem.split('diff_')[1].split('_')[0]
            runtype = PurePath(depth_diff_fn).stem.split(f'wy{WY}_')[1].split('_')[0]
            # Locate terrain file
            terrain_fn = h.fn_list(script_dir, f'{basin}*_setup/data/{basin}_terrain.nc')[0]
            if verbose:
                print(f'{terrain_fn}\n')

            # Load the terrain and SDD shift datasets
            if verbose:
                print('Loading datasets...')
            ds = xr.open_dataset(terrain_fn, drop_variables=['spatial_ref', 'hs', 'slope'])
            depth_diff_ds = xr.open_dataset(depth_diff_fn)

            # combine the two datasets
            combo_ds = xr.merge([ds, depth_diff_ds])
            # Derive dem elevation ranges from the terrain dataset using hundreds place rounding and flexible bin number
            p = None
            if verbose:
                print('Binning elevation data...')
            _, dem_elev_ranges = proc.bin_elev(dem=combo_ds['dem'], basinname=basin,
                                               plot_on=False, round_on=True, p=p, verbose=verbose)
            # Extract elevation bins from the ranges
            # Convert dict values into a flat list
            elevation_bins = [f[0] for jdx, f in enumerate(dem_elev_ranges.values())]
            # add the last bin (max elev)
            elevation_bins.append(list(dem_elev_ranges.values())[-1][-1])
            if verbose:
                print(f'{elevation_bins}\n')

            # Plot radial plots of snow depth differences
            num_aspect_bins = 8
            aspect_labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
            # if p is not None:
            #     outdir = f'{outdir}/radial_{num_aspect_bins}_p{p}'
            # else:
            #     outdir = f'{outdir}/radial_{num_aspect_bins}'
            # If directory doesn't exist, create it using pathlib
            Path(outdir).mkdir(parents=True, exist_ok=True)

            if p is not None:
                outname = f'{outdir}/{PurePath(depth_diff_fn).stem.split('_original')[0]}_radial_{num_aspect_bins}_p{p}.png'
            else:
                outname = f'{outdir}/{PurePath(depth_diff_fn).stem.split('_original')[0]}_radial_{num_aspect_bins}.png'
            # if verbose:
            print(f'Plotting radial plot to {outname}\n')
            title = f'{basin.capitalize()} {runtype} {dt}'
            cmap = sns.palettes.color_palette(palette, as_cmap=True)
            lim = 1.5
            if lim is not None:
                outname = outname.replace('.png', f'_lim{lim}.png')
            psr.plot_radial(combo_ds=combo_ds, combo_var='depth_diff', combo_var_units='m',
                            elevation_bins=elevation_bins,
                            cmap=cmap, num_aspect_bins=num_aspect_bins,
                            aspect_labels=aspect_labels,
                            elev_fontcolor="k", title=title,
                            vminnorm=-lim, vmaxnorm=lim, outname=outname,
                            figsize=(4, 3.25)
                            )

In [ ]:
basin = 'yampa'
wy = 2024
crank_snow_depth_diff_radial(basin, WY=wy, original=True, verbose=False)

In [ ]:
# Plot the diff histograms
workdir='/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/ASO_diffs'
script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
outdir='/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update/plot_eb_terms/figures/depth_diff'

# Locate snow depth difference files
depth_diff_fns = h.fn_list(workdir, f'{basin}*wy{wy}*diff*depth_original.nc')
# Only run for the iSnobal diffs removing fns if 'ua' or 'nwm' in filename
depth_diff_fns = [h for h in depth_diff_fns if 'ua' not in h and 'nwm' not in h]
depth_diff_fns

In [ ]:
depth_diffs = [h.load(diff_fn) for diff_fn in depth_diff_fns]
depth_diffs[0]

In [ ]:
da = depth_diffs[1]
cmap = 'PuOr'
vmin, vmax = -1.5, 1.5
cbar_label='snow depth difference (m)'
setfc = 'k'
title = ''
bins = 20
hist_xlabel = 'snow depth difference (m)'

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
h.plot_one(da, scale=None,
               title=title,
               cmap=cmap,
               setfc=setfc,
               specify_ax=(fig, ax),
               vmin=vmin, vmax=vmax,
               cbar_kwargs={'label': cbar_label},
               turnofflabels=True,
               turnoffaxes=True)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2))
h.plot_hist(da, color='teal', alpha=0.35, ec='k',
            bins=bins, range=(vmin, vmax),
            specify_ax=(fig, ax),
            xlabel=hist_xlabel,
            title=title)
# Increase the font size of the x and y ticks
ax.tick_params(axis='both', which='major', labelsize=12)
ax.axvline(0, color='black', linestyle='--')
stats_text = (
            f'n: {(~np.isnan(da.values)).sum()}\n'
            f'Mean: {float(da.mean().values):.2f}m\n'
            f'Median: {np.nanmedian(da.values):.2f}m\n'
            f'Std dev: {np.nanstd(da.values):.1f}'
        )
ax.text(0.95, 0.95, stats_text,
            transform=ax.transAxes, fontsize=12,
            verticalalignment='top', horizontalalignment='right',
            );
ax.set_ylim(0, 150000)
ax.set_xlabel('snow depth difference (m)', fontsize=12)


In [ ]:
# Quickly read in polygons and compute areas
poly_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys'
poly_fns = h.flatten([h.fn_list(poly_dir, f'*{basin}*.shp') for basin in basins])
poly_fns

In [ ]:
# Read them in as geodataframes and compute areas
import geopandas as gpd
polys = [gpd.read_file(poly_fn) for poly_fn in poly_fns]
for poly in polys:
    # Drop all except these columns
    keep_col = ['fid', 'objectid', 'areaacres', 'areasqkm', 'states', 'name',
                'hutype', 'layer', 'geometry', 'area_km2']
    poly.drop(columns=[col for col in poly.columns if col not in keep_col], inplace=True)
    poly['area_km2'] = poly.geometry.area / 1e6
    print(poly[['name', 'area_km2']])